# 03 — Document Ingestion Basics

## Objective

Understand how files become available for processing in Databricks: how paths and Volumes work, the difference between reading a file as **binary** versus as **structured/text** data, and what metadata Spark attaches to a file automatically.

This notebook reads the exact files `02_synthetic_data_generation.ipynb` wrote -- explicit, known paths only. It does **not** scan a Volume or workspace for arbitrary files; that would be unsafe in a real workspace and isn't the point of this notebook anyway.

## What We Will Learn

- How Unity Catalog Volumes expose files at a normal-looking filesystem path (`/Volumes/catalog/schema/volume/...`)
- The difference between reading files as **binary** (`binaryFile` format -- one row per file, raw bytes) and as **text/structured** (`text` / `csv` format -- one row per line or record)
- What metadata Spark captures for you when reading files (`path`, `modificationTime`, `length`) versus metadata you have to derive yourself
- Why the *same* underlying bytes look completely different depending on which reader you use -- and why that choice matters for the parsing step in Notebook 04

## Prerequisites

- Completed `02_synthetic_data_generation.ipynb` **with the same catalog/schema/volume widget values** -- this notebook reads the `.txt` and `.csv` files it wrote, and will fail with a "path not found" error otherwise
- A cluster or SQL warehouse attached to this notebook

## Conceptual Explanation

**Volumes as file paths.** A Unity Catalog Volume mounts governed storage at a path that behaves like a normal filesystem: `/Volumes/<catalog>/<schema>/<volume>/...`. You can list it, read from it, and write to it with `dbutils.fs`, Spark file readers, or (on most clusters) plain Python `open()`. This is what lets Databricks treat cloud object storage like local files.

**Binary vs. structured reading.** The same file can be read two fundamentally different ways:
- `spark.read.format("binaryFile")` treats each *file* as one row: `path`, `modificationTime`, `length`, and `content` (the raw bytes, untouched). This works for **any** file type -- PDF, image, `.txt`, whatever -- because it doesn't try to interpret the bytes at all. This is the format `ai_parse_document` (Notebook 04) expects as input for binary formats like PDF.
- `spark.read.text(...)` or `spark.read.csv(...)` treat the file's *content* as structured: one row per line, or one row per CSV record with typed columns. Spark has to understand the file's structure to do this.

We only generated `.txt` and `.csv` files in Notebook 02 (no PDF), so everything below runs against real files. `binaryFile` reads them exactly the same way it would read a PDF or image -- the format doesn't care what's inside. When Notebook 04 uses `ai_parse_document`, it typically starts from this same `binaryFile` representation.

**Metadata Spark gives you for free vs. metadata you derive.** `binaryFile` reads hand you `path`, `length` (bytes), and `modificationTime` automatically. Anything else -- like "which document title does this file belong to" -- has to be derived from the path or content yourself, which we'll do below.

## Example Data

The files Notebook 02 wrote to the Volume:
- `<volume>/documents/*.txt` -- 15 fictional policy/procedure/product files
- `<volume>/transactions/transactions.csv` -- one CSV file with ~250 fictional transaction rows

No new data is generated in this notebook.

## Implementation

### Step 1 — Point at the same catalog/schema/volume as Notebook 02

These defaults must match what you used in `02_synthetic_data_generation.ipynb`, or the paths below won't exist.

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")
dbutils.widgets.text("volume_name", "synthetic_data", "Volume")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

documents_path = f"{volume_path}/documents"
transactions_csv_path = f"{volume_path}/transactions/transactions.csv"

print(f"Documents folder: {documents_path}")
print(f"Transactions CSV: {transactions_csv_path}")

### Step 2 — List the folder and look at raw file metadata

Note this lists exactly one, explicit, known folder -- not the whole Volume, and not the workspace.

In [ ]:
files = dbutils.fs.ls(documents_path)
for f in files[:5]:
    print(f"{f.name:45s} {f.size:6d} bytes")
print(f"... {len(files)} files total")

### Step 3 — Read the documents folder as binary

One row per file, raw bytes untouched. This is the representation any file type -- including a PDF -- would take.

In [ ]:
binary_df = spark.read.format("binaryFile").load(documents_path)
binary_df.printSchema()
display(binary_df.select("path", "modificationTime", "length"))

The `content` column is raw `bytes`. Decoding it is *your* job -- Spark won't do it for you, because it has no idea whether the bytes are text, a PDF, or an image.

In [ ]:
first_file = binary_df.select("path", "content").first()
decoded_text = first_file["content"].decode("utf-8")
print(f"File: {first_file['path']}\n")
print(decoded_text)

### Step 4 — Read the same folder as text

By default, `spark.read.text(...)` gives **one row per line**, not one row per file -- very different from `binaryFile`.

In [ ]:
text_lines_df = spark.read.text(documents_path)
print(f"binaryFile row count (one per file): {binary_df.count()}")
print(f"text() row count (one per line):     {text_lines_df.count()}")
display(text_lines_df.limit(8))

Passing `wholetext=True` switches it back to one row per file -- but still typed as a single text `value` column, not raw bytes like `binaryFile`.

In [ ]:
text_whole_df = spark.read.text(documents_path, wholetext=True)
print(f"text(wholetext=True) row count: {text_whole_df.count()}")
text_whole_df.printSchema()

### Step 5 — Read the CSV as structured data

`inferSchema=True` makes Spark scan the data once to guess column types. Compare the inferred types below with the `transactions` Delta table from Notebook 02 -- they may not match exactly, since a CSV file has no native type information.

In [ ]:
transactions_csv_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(transactions_csv_path)
)
transactions_csv_df.printSchema()
display(transactions_csv_df.limit(5))

In [ ]:
%sql
-- Update the catalog/schema below if you changed the widgets in Step 1
DESCRIBE main.genai_lab.transactions

### Step 6 — Derive metadata the `binaryFile` reader doesn't give you

`binaryFile` knows `path`, `length`, and `modificationTime` -- it has no idea this file's `doc_id` is `1` or its title is "Checking Account Overview". Deriving that requires either parsing the filename (what we do here) or joining back to the `documents` table from Notebook 02 using the filename you generated it with.

In [ ]:
from pyspark.sql.functions import element_at, split, regexp_extract

file_manifest_df = (
    binary_df
    .withColumn("file_name", element_at(split("path", "/"), -1))
    .withColumn("doc_id", regexp_extract("file_name", r"^(\d+)_", 1).cast("int"))
    .select("doc_id", "file_name", "length", "modificationTime")
    .orderBy("doc_id")
)
display(file_manifest_df)

## Inspect the Output

- Compare the two row counts printed in Step 4 -- confirm you understand *why* they differ (one file vs. many lines).
- Check the CSV schema from Step 5 against `DESCRIBE main.genai_lab.transactions` -- note any type differences (e.g. a column Delta stored as `date` may come back as `string` or `timestamp` from CSV inference).
- In `file_manifest_df`, confirm `doc_id` extracted correctly for all 15 files and lines up with the `documents` table's `doc_id` column from Notebook 02.

## Experimentation Section

1. Read a **single explicit file** instead of a folder -- e.g. `spark.read.format("binaryFile").load(f"{documents_path}/01_checking_account_overview.txt")` -- and confirm you get exactly one row.
2. Drop `.option("header", True)` from the CSV read in Step 5 and see how the schema and first row change.
3. Try `spark.read.csv(transactions_csv_path)` without `inferSchema` at all -- what type does every column get by default?
4. Join `file_manifest_df` back to the `documents` Delta table on `doc_id` and confirm `file_name` and `title` line up for every row.
5. Try pointing `documents_path` at a folder that doesn't exist (e.g. add `/nonexistent` to the path) and read the resulting error message carefully -- this is the error you'll see if Notebook 02 wasn't run with matching widget values.

## Common Errors / Limitations

- **`PATH_NOT_FOUND`** -- almost always means the catalog/schema/volume widgets here don't match what you used in Notebook 02. Re-check them before anything else.
- **Never point these readers at a broad or unknown path** (e.g. the root of a shared production Volume) just to "see what's there." Always use an explicit, known path, as this notebook does -- an unscoped scan can be slow, expensive, or read data you weren't meant to touch.
- **`content.decode("utf-8")` will raise on non-text bytes** -- it works here because our files are plain text; it would fail on a real PDF's bytes, which need a parser (Notebook 04), not a text decode.
- **`inferSchema=True` reads the file twice** (once to infer types, once to load) -- harmless at this size, but worth knowing before using it on a large real CSV.
- **This notebook does not call `ai_parse_document`** -- it only prepares the binary/text representation that a real parser would consume next.

## Summary

You read the same folder of files three different ways -- as opaque binary (`binaryFile`), as line-oriented text (`text`), and as typed structured data (`csv`) -- and saw how the row count and schema change with each. You also built a small file manifest from metadata Spark gives you for free plus metadata you have to derive yourself. This is exactly the binary representation Notebook 04 starts from when it introduces `ai_parse_document`.

## Suggested Exercises

- Write a one-line summary of when you'd reach for `binaryFile` vs. `text` vs. `csv` in your own words.
- Look up the `multiLine` option for `spark.read.csv` and think about when a transactions export might need it (hint: free-text fields containing commas or newlines).
- When you're ready, move on to **`04_document_parsing_ai_parse_document.ipynb`**, which takes the `binaryFile` representation from Step 3 and runs it through actual document parsing.